# Experiment No. 4
## Text-to-Image Generation using a Pre-trained Diffusion Model
**Model:** `stabilityai/stable-diffusion-xl-base-1.0` | **Libraries:** Hugging Face Diffusers, PyTorch, Gradio

> Before running: **Runtime → Change runtime type → T4 GPU**

### Step 1: Install libraries

In [ ]:
!pip install -q diffusers transformers accelerate safetensors gradio

### Step 2: Import libraries and check GPU

In [ ]:
import torch
from diffusers import DiffusionPipeline
import gradio as gr

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
device = 'cuda' if torch.cuda.is_available() else 'cpu'

### Step 3: Load the pre-trained SDXL model (float16 on GPU)

In [ ]:
pipe = DiffusionPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    use_safetensors=True,
    variant='fp16' if device == 'cuda' else None,
)
pipe = pipe.to(device)
# pipe.enable_attention_slicing()  # uncomment if you hit GPU out-of-memory
print('Model loaded on', device)

### Step 4: Image generation function

In [ ]:
def generate_image(prompt, negative_prompt='', steps=30, guidance=7.5):
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt or None,
        num_inference_steps=int(steps),
        guidance_scale=float(guidance),
    )
    return result.images[0]

### Step 5: Test with different prompts

In [ ]:
prompts = [
    'A futuristic city at sunset, cinematic lighting, highly detailed',
    'A cute red panda reading a book in a cozy library, watercolor style',
    'A snowy mountain village at night, warm lights, ultra realistic, 4k',
]
for p in prompts:
    print('Prompt:', p)
    display(generate_image(p))

### Step 6: Effect of generation parameters
Same prompt with different **inference steps** and **guidance scale**.

In [ ]:
import matplotlib.pyplot as plt

test_prompt = 'A majestic lion standing on a cliff at sunrise, highly detailed'

# Effect of inference steps (guidance fixed)
step_values = [10, 30, 50]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, s in zip(axes, step_values):
    ax.imshow(generate_image(test_prompt, steps=s, guidance=7.5)); ax.set_title(f'Steps = {s}'); ax.axis('off')
plt.show()

# Effect of guidance scale (steps fixed)
guidance_values = [3, 7.5, 12]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, g in zip(axes, guidance_values):
    ax.imshow(generate_image(test_prompt, steps=30, guidance=g)); ax.set_title(f'Guidance = {g}'); ax.axis('off')
plt.show()

### Step 7: Gradio interface

In [ ]:
demo = gr.Interface(
    fn=generate_image,
    inputs=[
        gr.Textbox(label='Prompt', lines=2, placeholder='A futuristic city at sunset, cinematic lighting'),
        gr.Textbox(label='Negative Prompt (optional)', placeholder='blurry, low quality'),
        gr.Slider(10, 50, value=30, step=1, label='Inference Steps'),
        gr.Slider(1, 15, value=7.5, step=0.5, label='Guidance Scale'),
    ],
    outputs=gr.Image(label='Generated Image'),
    title='Text-to-Image Generation using Stable Diffusion XL',
    description='Enter a prompt, adjust the parameters and click Submit.',
    flagging_mode='never',  # use allow_flagging='never' on older Gradio
)
demo.launch(share=True, debug=True)

### Step 8 (optional): Save the model for a Flask app
The saved folder is about 7 GB.

In [ ]:
pipe.save_pretrained('./local_model/sdxl_base', safe_serialization=True)
print('Saved to ./local_model/sdxl_base')

In [ ]:
# Zip and download (takes a few minutes)
!zip -r -q sdxl_base.zip local_model/sdxl_base
from google.colab import files
files.download('sdxl_base.zip')

### Observations / Conclusion
- More inference steps generally give finer detail, with diminishing returns after about 30–40 steps.
- Low guidance gives creative but loosely matching images; high guidance follows the prompt closely but can look over-saturated.
- Detailed prompts (subject, style, lighting, composition) give better results.

The experiment demonstrated text-to-image generation with a pre-trained diffusion model, prompt engineering, and a Gradio interface.